# Exploración inicial de datos — StreamView Analytics

## Evaluación Parcial 1

Esta exploración tiene como propósito comprender la estructura de las fuentes de información antes de realizar limpieza, transformación, KPIs o visualizaciones. Cada resultado responde una pregunta concreta del caso y se limita a observar, validar y documentar.

### Propósito

- Verificar qué información y variables contiene cada fuente.
- Identificar qué variables son compartidas, exclusivas y comparables.
- Contrastar la estructura observada con el diccionario del caso.
- Detectar señales que deberán evaluarse posteriormente en `feature/calidad-datos` y `feature/preparacion-catalogo`.

Los datasets RAW se cargan en modo lectura y no se modifican en este notebook.

## Preguntas guía

### Sobre las fuentes

- ¿Cuántos registros y variables contiene cada dataset?
- ¿Qué representa cada registro?
- ¿Qué columnas posee cada fuente?
- ¿Qué variables comparten Movies y TV Shows, y cuáles son exclusivas?

### Sobre el caso y EP1

- ¿La estructura observada coincide con la documentación del caso?
- ¿Están disponibles las variables necesarias para los análisis posteriores?
- ¿Qué variables pueden compararse entre Movies y TV Shows?
- ¿Qué diferencias de tipo o estructura requieren análisis posterior?

### Sobre la preparación futura

- ¿Qué tipos de datos se observan?
- ¿Existen señales de formatos o variables multivalor que deban revisarse?
- ¿Qué aspectos deben trasladarse a la etapa de calidad de datos?

## Importaciones

Se utilizan únicamente herramientas para rutas, carga tabular y construcción de tablas de validación. No se aplican transformaciones persistentes ni librerías de modelamiento.

In [49]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)

## Configuración de rutas

La raíz se resuelve de manera portátil: funciona tanto al ejecutar el notebook desde `notebooks/` como desde la raíz del repositorio. Antes de leer, se valida la existencia de los dos archivos requeridos.

In [50]:
working_directory = Path.cwd().resolve()
PROJECT_ROOT = (
    working_directory
    if (working_directory / "data" / "raw").exists()
    else working_directory.parent
)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
MOVIES_PATH = DATA_RAW / "netflix_movies_detailed_up_to_2025.csv"
TV_SHOWS_PATH = DATA_RAW / "netflix_tv_shows_detailed_up_to_2025.csv"

for dataset_path in (MOVIES_PATH, TV_SHOWS_PATH):
    if not dataset_path.is_file():
        raise FileNotFoundError(
            f"No se encontró el archivo requerido: {dataset_path}. "
            "Verifica que los CSV estén en data/raw/."
        )

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Directorio RAW: {DATA_RAW}")

Raíz del proyecto: C:\Users\cesar\OneDrive\Desktop\DuocUC\3erYear\visualizacionDatos\streamview-analytics-visualizacion-datos
Directorio RAW: C:\Users\cesar\OneDrive\Desktop\DuocUC\3erYear\visualizacionDatos\streamview-analytics-visualizacion-datos\data\raw


## Carga de datasets

Se cargan los CSV originales en memoria bajo los nombres `movies_df` y `tv_shows_df`. Esta carga no modifica los archivos RAW.

In [51]:
movies_df = pd.read_csv(MOVIES_PATH)
tv_shows_df = pd.read_csv(TV_SHOWS_PATH)

print("Datasets cargados correctamente.")

Datasets cargados correctamente.


## 1. Validación de dimensiones

Esta tabla responde cuántos registros y variables tiene cada fuente. El caso describe aproximadamente 16.000 registros por dataset; la comparación se realiza contra esa referencia, sin asumir de antemano un valor exacto.

In [52]:
dimension_summary = pd.DataFrame(
    {
        "fuente": ["Movies", "TV Shows"],
        "registros": [movies_df.shape[0], tv_shows_df.shape[0]],
        "variables": [movies_df.shape[1], tv_shows_df.shape[1]],
        "referencia_del_caso": ["≈ 16.000 registros", "≈ 16.000 registros"],
    }
)

dimension_summary

,fuente,registros,variables,referencia_del_caso
0,Movies,16000,18,≈ 16.000 registros
1,TV Shows,16000,16,≈ 16.000 registros


In [53]:
for row in dimension_summary.itertuples(index=False):
    diferencia_referencia = row.registros - 16_000
    print(
        f"{row.fuente}: {row.registros:,} registros y {row.variables} variables "
        f"(diferencia frente a 16.000: {diferencia_referencia:+,})."
    )

Movies: 16,000 registros y 18 variables (diferencia frente a 16.000: +0).
TV Shows: 16,000 registros y 16 variables (diferencia frente a 16.000: +0).


## 2. Inspección de registros

Se revisan tres registros por fuente para comprender visualmente la unidad de observación y los campos disponibles. Esta muestra no se usa para inferir distribuciones ni resultados de negocio.

In [54]:
movies_df.head(3)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,language,description,popularity,vote_count,vote_average,budget,revenue
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antonio Banderas, Walt Dohrn",United States of America,2010-05-16,2010,6.380,NaN,"Comedy, Adventure, Fantasy, Animation, Family",en,"A bored and domesticated Shrek pacts with deal-maker Rumpelstiltskin to get back to feeling like a real ogre again, ...",203.893,7449,6.380,165000000,752600867
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken Watanabe, Tom Hardy, Elliot Page","United Kingdom, United States of America",2010-07-15,2010,8.369,NaN,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a c...",156.242,37119,8.369,160000000,839030630
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, Toby Jones, Helena Bonham Carter","United Kingdom, United States of America",2010-11-17,2010,7.744,NaN,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their last year at Hogwarts to find and destroy the remaining Horcruxes, putt...",121.191,19327,7.744,250000000,954305868


In [55]:
tv_shows_df.head(3)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,language,description,popularity,vote_count,vote_average
0,33238,TV Show,Running Man,안재철,"Yoo Jae-suk, Jee Seok-jin, Kim Jong-kook, Haha, Song Ji-hyo",South Korea,2010-07-11,2010,8.241,1 Seasons,"Comedy, Reality",ko,A reality and competition show where members are joined by celebs to complete a weekly mission to win the race with ...,1929.898,187,8.241
1,32415,TV Show,Conan,NaN,"Conan O'Brien, Andy Richter",United States of America,2010-11-08,2010,7.035,1 Seasons,"Talk, Comedy, News",en,A late night television talk show hosted by Conan O'Brien.,1670.580,229,7.035
2,37757,TV Show,MasterChef Greece,NaN,NaN,Greece,2010-10-03,2010,5.600,1 Seasons,Reality,el,MasterChef Greece is a Greek competitive cooking game show. It's an adaptation of the British show MasterChef.,1317.092,6,5.600


**Interpretación:** la muestra permite observar si cada fila reúne atributos de un contenido audiovisual. La validación de identificadores se revisa más adelante para complementar esta observación.

## 3. Comparación de columnas

Se identifican columnas comunes y exclusivas. La presencia en ambas fuentes es una condición necesaria —pero no suficiente— para comparar una variable entre Movies y TV Shows; sus tipos y significado también se revisan.

In [56]:
columnas_movies = set(movies_df.columns)
columnas_tv = set(tv_shows_df.columns)

columnas_comunes = sorted(columnas_movies & columnas_tv)
solo_movies = sorted(columnas_movies - columnas_tv)
solo_tv = sorted(columnas_tv - columnas_movies)

column_presence = pd.DataFrame(
    {
        "variable": sorted(columnas_movies | columnas_tv),
        "movies": [column in columnas_movies for column in sorted(columnas_movies | columnas_tv)],
        "tv_shows": [column in columnas_tv for column in sorted(columnas_movies | columnas_tv)],
    }
)

print(f"Columnas comunes ({len(columnas_comunes)}): {columnas_comunes}")
print(f"Solo Movies ({len(solo_movies)}): {solo_movies}")
print(f"Solo TV Shows ({len(solo_tv)}): {solo_tv}")

column_presence

Columnas comunes (16): ['cast', 'country', 'date_added', 'description', 'director', 'duration', 'genres', 'language', 'popularity', 'rating', 'release_year', 'show_id', 'title', 'type', 'vote_average', 'vote_count']
Solo Movies (2): ['budget', 'revenue']
Solo TV Shows (0): []


,variable,movies,tv_shows
0,budget,True,False
1,cast,True,True
2,country,True,True
3,date_added,True,True
4,description,True,True
5,director,True,True
6,duration,True,True
7,genres,True,True
8,language,True,True
9,popularity,True,True


## 4. Validación preliminar con el diccionario del caso

La siguiente tabla contrasta las variables esperadas con la estructura real y muestra el tipo observado en cada fuente. Es una validación de disponibilidad, no una decisión de limpieza.

In [57]:
variables_compartidas_esperadas = [
    "show_id",
    "type",
    "title",
    "director",
    "cast",
    "country",
    "date_added",
    "release_year",
    "rating",
    "duration",
    "genres",
    "language",
    "description",
    "popularity",
    "vote_average",
    "vote_count",
]
variables_exclusivas_movies_esperadas = ["budget", "revenue"]
variables_esperadas = variables_compartidas_esperadas + variables_exclusivas_movies_esperadas

referencia_caso = {
    "show_id": ("texto", "identificador del contenido"),
    "type": ("categórica", "tipo de contenido audiovisual"),
    "title": ("no especificado", "título del contenido"),
    "director": ("no especificado", "dirección del contenido"),
    "cast": ("no especificado", "reparto del contenido"),
    "country": ("no especificado", "país o países asociados"),
    "date_added": ("fecha", "fecha de incorporación al catálogo"),
    "release_year": ("no especificado", "año de estreno"),
    "rating": ("categórica", "clasificación etaria según el caso"),
    "duration": ("representación de duración", "duración del contenido"),
    "genres": ("no especificado", "género o géneros del contenido"),
    "language": ("no especificado", "idioma del contenido"),
    "description": ("no especificado", "descripción del contenido"),
    "popularity": ("índice numérico relativo", "índice relativo; no representa reproducciones"),
    "vote_average": ("numérica 0–10", "calificación promedio del público"),
    "vote_count": ("no especificado", "cantidad de votos"),
    "budget": ("numérica financiera", "presupuesto; exclusivo de Movies"),
    "revenue": ("numérica financiera", "ingresos; exclusivo de Movies"),
}

def observed_dtype(dataframe: pd.DataFrame, column: str) -> object:
    return str(dataframe[column].dtype) if column in dataframe.columns else pd.NA

def dictionary_observation(column: str) -> str:
    movies_type = observed_dtype(movies_df, column)
    tv_type = observed_dtype(tv_shows_df, column)
    if column == "show_id" and (movies_type != "object" or tv_type != "object"):
        return "El caso lo documenta como texto; la conveniencia de representarlo como texto debe evaluarse después."
    if column == "rating" and movies_type != "object" and tv_type != "object":
        return "El caso lo define como categórica; la representación numérica requiere validación dirigida."
    if column == "date_added" and (movies_type != "datetime64[ns]" or tv_type != "datetime64[ns]"):
        return "El caso la describe como fecha; el formato observado debe revisarse posteriormente."
    if column in {"budget", "revenue"}:
        return "Debe estar disponible solo en Movies; se valida su presencia en la tabla."
    return "La presencia y el tipo observado se documentan; su comparabilidad requiere revisión posterior."

dictionary_validation = pd.DataFrame(
    {
        "variable": variables_esperadas,
        "tipo_esperado_segun_caso": [referencia_caso[column][0] for column in variables_esperadas],
        "significado_esperado": [referencia_caso[column][1] for column in variables_esperadas],
        "presente_movies": [column in columnas_movies for column in variables_esperadas],
        "presente_tv": [column in columnas_tv for column in variables_esperadas],
        "tipo_movies": [observed_dtype(movies_df, column) for column in variables_esperadas],
        "tipo_tv": [observed_dtype(tv_shows_df, column) for column in variables_esperadas],
        "observacion_preliminar": [dictionary_observation(column) for column in variables_esperadas],
    }
)

dictionary_validation

,variable,tipo_esperado_segun_caso,significado_esperado,presente_movies,presente_tv,tipo_movies,tipo_tv,observacion_preliminar
0,show_id,texto,identificador del contenido,True,True,int64,int64,El caso lo documenta como texto; la conveniencia de representarlo como texto debe evaluarse después.
1,type,categórica,tipo de contenido audiovisual,True,True,str,str,La presencia y el tipo observado se documentan; su comparabilidad requiere revisión posterior.
2,title,no especificado,título del contenido,True,True,str,str,La presencia y el tipo observado se documentan; su comparabilidad requiere revisión posterior.
3,director,no especificado,dirección del contenido,True,True,str,str,La presencia y el tipo observado se documentan; su comparabilidad requiere revisión posterior.
4,cast,no especificado,reparto del contenido,True,True,str,str,La presencia y el tipo observado se documentan; su comparabilidad requiere revisión posterior.
5,country,no especificado,país o países asociados,True,True,str,str,La presencia y el tipo observado se documentan; su comparabilidad requiere revisión posterior.
6,date_added,fecha,fecha de incorporación al catálogo,True,True,str,str,El caso la describe como fecha; el formato observado debe revisarse posteriormente.
7,release_year,no especificado,año de estreno,True,True,int64,int64,La presencia y el tipo observado se documentan; su comparabilidad requiere revisión posterior.
8,rating,categórica,clasificación etaria según el caso,True,True,float64,float64,El caso lo define como categórica; la representación numérica requiere validación dirigida.
9,duration,representación de duración,duración del contenido,True,True,float64,str,La presencia y el tipo observado se documentan; su comparabilidad requiere revisión posterior.


## 5. Inspección de tipos de datos

Se revisan los tipos de las variables relevantes para EP1. Cualquier diferencia detectada se documenta como una pregunta para calidad de datos; no se convierte ningún tipo en esta etapa.

In [58]:
variables_foco = [
    "show_id",
    "type",
    "date_added",
    "release_year",
    "duration",
    "popularity",
    "vote_average",
    "vote_count",
    "budget",
    "revenue",
]

dtype_comparison = pd.DataFrame(
    {
        "variable": variables_foco,
        "tipo_movies": [observed_dtype(movies_df, column) for column in variables_foco],
        "tipo_tv": [observed_dtype(tv_shows_df, column) for column in variables_foco],
        "comparable_por_presencia": [
            column in columnas_movies and column in columnas_tv
            for column in variables_foco
        ],
    }
)

dtype_comparison

,variable,tipo_movies,tipo_tv,comparable_por_presencia
0,show_id,int64,int64,True
1,type,str,str,True
2,date_added,str,str,True
3,release_year,int64,int64,True
4,duration,float64,str,True
5,popularity,float64,float64,True
6,vote_average,float64,float64,True
7,vote_count,int64,int64,True
8,budget,int64,NaN,False
9,revenue,int64,NaN,False


## 6. Disponibilidad y formato de `date_added`

El caso define `date_added` como fecha de incorporación y la EP1 contempla análisis temporal. Se revisa su disponibilidad y, solo de forma diagnóstica, si los valores no nulos pueden interpretarse como fechas. La columna original no se modifica.

In [59]:
def date_added_validation(dataframe: pd.DataFrame, source_name: str) -> dict[str, object]:
    if "date_added" not in dataframe.columns:
        return {
            "fuente": source_name,
            "disponible": False,
            "tipo_observado": pd.NA,
            "total_registros": len(dataframe),
            "no_nulos": pd.NA,
            "nulos": pd.NA,
            "porcentaje_nulo": pd.NA,
            "interpretable_como_fecha": pd.NA,
            "no_interpretable": pd.NA,
            "ejemplos_formato": pd.NA,
        }

    date_series = dataframe["date_added"]
    parsed_for_diagnosis = pd.to_datetime(date_series, errors="coerce")
    non_null = int(date_series.notna().sum())
    interpretable = int(parsed_for_diagnosis.notna().sum())
    return {
        "fuente": source_name,
        "disponible": True,
        "tipo_observado": str(date_series.dtype),
        "total_registros": len(date_series),
        "no_nulos": non_null,
        "nulos": int(date_series.isna().sum()),
        "porcentaje_nulo": date_series.isna().mean() * 100,
        "interpretable_como_fecha": interpretable,
        "no_interpretable": non_null - interpretable,
        "ejemplos_formato": "; ".join(str(value) for value in date_series.head(3)),
    }

date_added_validation_table = pd.DataFrame(
    [
        date_added_validation(movies_df, "Movies"),
        date_added_validation(tv_shows_df, "TV Shows"),
    ]
)

date_added_validation_table

,fuente,disponible,tipo_observado,total_registros,no_nulos,nulos,porcentaje_nulo,interpretable_como_fecha,no_interpretable,ejemplos_formato
0,Movies,True,str,16000,16000,0,0.0,16000,0,2010-05-16; 2010-07-15; 2010-11-17
1,TV Shows,True,str,16000,16000,0,0.0,16000,0,2010-07-11; 2010-11-08; 2010-10-03


## 7. Disponibilidad y rango de `release_year`

Esta validación confirma si `release_year` existe y está estructuralmente disponible para análisis temporal posterior. No se agrupa contenido ni se infieren tendencias.

In [60]:
def release_year_validation(dataframe: pd.DataFrame, source_name: str) -> dict[str, object]:
    if "release_year" not in dataframe.columns:
        return {
            "fuente": source_name,
            "disponible": False,
            "tipo_observado": pd.NA,
            "nulos": pd.NA,
            "valores_unicos": pd.NA,
            "minimo": pd.NA,
            "maximo": pd.NA,
        }

    year_series = dataframe["release_year"]
    return {
        "fuente": source_name,
        "disponible": True,
        "tipo_observado": str(year_series.dtype),
        "nulos": int(year_series.isna().sum()),
        "valores_unicos": int(year_series.nunique(dropna=True)),
        "minimo": year_series.min(),
        "maximo": year_series.max(),
    }

release_year_validation_table = pd.DataFrame(
    [
        release_year_validation(movies_df, "Movies"),
        release_year_validation(tv_shows_df, "TV Shows"),
    ]
)

release_year_validation_table

,fuente,disponible,tipo_observado,nulos,valores_unicos,minimo,maximo
0,Movies,True,int64,0,16,2010,2025
1,TV Shows,True,int64,0,16,2010,2025


## 8. Validación dirigida de `rating` y `vote_average`

El caso define `rating` como clasificación etaria/categórica y `vote_average` como valoración promedio en escala 0–10. Esta comprobación contrasta ambas columnas sin corregir, eliminar ni reemplazar valores.

In [61]:
def rating_vote_validation(dataframe: pd.DataFrame, source_name: str) -> dict[str, object]:
    rating_series = dataframe["rating"]
    vote_series = dataframe["vote_average"]
    both_available = rating_series.notna() & vote_series.notna()
    matches = rating_series.eq(vote_series) & both_available
    comparable_records = int(both_available.sum())
    matching_records = int(matches.sum())
    percentage = (matching_records / comparable_records * 100) if comparable_records else np.nan
    return {
        "fuente": source_name,
        "tipo_rating": str(rating_series.dtype),
        "tipo_vote_average": str(vote_series.dtype),
        "ejemplos_rating": rating_series.head(3).tolist(),
        "ejemplos_vote_average": vote_series.head(3).tolist(),
        "valores_unicos_rating": int(rating_series.nunique(dropna=True)),
        "registros_comparables": comparable_records,
        "coincidencias": matching_records,
        "porcentaje_coincidencia": percentage,
    }

rating_validation = pd.DataFrame(
    [
        rating_vote_validation(movies_df, "Movies"),
        rating_vote_validation(tv_shows_df, "TV Shows"),
    ]
)

rating_validation

,fuente,tipo_rating,tipo_vote_average,ejemplos_rating,ejemplos_vote_average,valores_unicos_rating,registros_comparables,coincidencias,porcentaje_coincidencia
0,Movies,float64,float64,"[6.38, 8.369, 7.744]","[6.38, 8.369, 7.744]",2145,16000,16000,100.0
1,TV Shows,float64,float64,"[8.241, 7.035, 5.6]","[8.241, 7.035, 5.6]",1184,16000,16000,100.0


## 9. Disponibilidad real de `duration`

La presencia de una columna no garantiza información útil. Se revisan nulos, valores no nulos, variabilidad y ejemplos representativos sin transformar la variable.

In [62]:
def duration_validation(dataframe: pd.DataFrame, source_name: str) -> dict[str, object]:
    duration_series = dataframe["duration"]
    total_records = len(duration_series)
    null_records = int(duration_series.isna().sum())
    non_null_records = int(duration_series.notna().sum())
    examples = (
        "Sin valores no nulos"
        if non_null_records == 0
        else "; ".join(str(value) for value in duration_series.head(3))
    )
    return {
        "fuente": source_name,
        "tipo_observado": str(duration_series.dtype),
        "total_registros": total_records,
        "nulos": null_records,
        "no_nulos": non_null_records,
        "porcentaje_nulo": null_records / total_records * 100,
        "valores_unicos_no_nulos": int(duration_series.nunique(dropna=True)),
        "ejemplos": examples,
    }

duration_validation_table = pd.DataFrame(
    [
        duration_validation(movies_df, "Movies"),
        duration_validation(tv_shows_df, "TV Shows"),
    ]
)

duration_validation_table

,fuente,tipo_observado,total_registros,nulos,no_nulos,porcentaje_nulo,valores_unicos_no_nulos,ejemplos
0,Movies,float64,16000,16000,0,100.0,0,Sin valores no nulos
1,TV Shows,str,16000,0,16000,0.0,1,1 Seasons; 1 Seasons; 1 Seasons


In [63]:
# Diagnóstico de variabilidad de duration en TV Shows; no modifica la fuente.
tv_shows_df["duration"].value_counts(dropna=False)

duration
1 Seasons    16000
Name: count, dtype: int64

## 10. Validación de la escala de `vote_average`

El caso establece que `vote_average` pertenece al intervalo 0–10. Se verifica esa regla de negocio mediante mínimo, máximo, nulos y conteo de valores fuera de rango; no se analizan promedios ni se construyen gráficos.

In [64]:
def vote_average_range_validation(dataframe: pd.DataFrame, source_name: str) -> dict[str, object]:
    votes = dataframe["vote_average"]
    outside_range = (votes.lt(0) | votes.gt(10)).sum()
    return {
        "fuente": source_name,
        "minimo": votes.min(),
        "maximo": votes.max(),
        "nulos": int(votes.isna().sum()),
        "fuera_rango_0_10": int(outside_range),
    }

vote_average_validation = pd.DataFrame(
    [
        vote_average_range_validation(movies_df, "Movies"),
        vote_average_range_validation(tv_shows_df, "TV Shows"),
    ]
)

vote_average_validation

,fuente,minimo,maximo,nulos,fuera_rango_0_10
0,Movies,0.0,10.0,0,0
1,TV Shows,0.0,10.0,0,0


## 11. Validación preliminar de `type`

La regla del caso indica que Movies debe representar `Movie` y TV Shows debe representar `TV Show`. Se muestran los conteos incluidos valores faltantes, sin corregir ni filtrar registros.

In [65]:
def value_counts_table(dataframe: pd.DataFrame, source_name: str, column: str) -> pd.DataFrame:
    if column not in dataframe.columns:
        return pd.DataFrame(
            {"fuente": [source_name], "valor": [pd.NA], "cantidad": [pd.NA]}
        )

    counts = dataframe[column].value_counts(dropna=False).rename_axis("valor").reset_index(name="cantidad")
    counts.insert(0, "fuente", source_name)
    return counts

type_validation = pd.concat(
    [
        value_counts_table(movies_df, "Movies", "type"),
        value_counts_table(tv_shows_df, "TV Shows", "type"),
    ],
    ignore_index=True,
)

type_validation

,fuente,valor,cantidad
0,Movies,Movie,16000
1,TV Shows,TV Show,16000


## 12. Revisión preliminar de `show_id`

Se compara el número de filas con la cantidad de identificadores únicos. La existencia de una diferencia no se corrige aquí: se registra como un aspecto que debe investigar la rama de calidad de datos.

In [66]:
def identifier_summary(dataframe: pd.DataFrame, source_name: str) -> dict[str, object]:
    if "show_id" not in dataframe.columns:
        return {
            "fuente": source_name,
            "registros": len(dataframe),
            "tipo_esperado_segun_caso": "texto",
            "tipo_observado": pd.NA,
            "show_id_unicos": pd.NA,
            "diferencia_registros_vs_unicos": pd.NA,
        }

    unique_ids = dataframe["show_id"].nunique(dropna=True)
    return {
        "fuente": source_name,
        "registros": len(dataframe),
        "tipo_esperado_segun_caso": "texto",
        "tipo_observado": str(dataframe["show_id"].dtype),
        "show_id_unicos": unique_ids,
        "diferencia_registros_vs_unicos": len(dataframe) - unique_ids,
    }

show_id_validation = pd.DataFrame(
    [
        identifier_summary(movies_df, "Movies"),
        identifier_summary(tv_shows_df, "TV Shows"),
    ]
)

show_id_validation

,fuente,registros,tipo_esperado_segun_caso,tipo_observado,show_id_unicos,diferencia_registros_vs_unicos
0,Movies,16000,texto,int64,16000,0
1,TV Shows,16000,texto,int64,15991,9


## 13. Variables necesarias para EP1

Se valida que las variables requeridas para caracterización, popularidad, valoración y análisis financiero estén disponibles en la fuente correspondiente. La disponibilidad no implica todavía que los valores estén completos o listos para usar.

In [67]:
variables_ep1 = [
    "type",
    "country",
    "release_year",
    "genres",
    "language",
    "popularity",
    "vote_average",
    "vote_count",
    "budget",
    "revenue",
]

ep1_availability = pd.DataFrame(
    {
        "variable": variables_ep1,
        "disponible_movies": [column in columnas_movies for column in variables_ep1],
        "disponible_tv_shows": [column in columnas_tv for column in variables_ep1],
        "uso_previsto": [
            "comparación de contenido",
            "análisis geográfico",
            "evolución temporal",
            "caracterización por género",
            "caracterización por idioma",
            "índice relativo de popularidad",
            "valoración en escala 0–10",
            "respaldo de la valoración",
            "análisis financiero solo Movies",
            "análisis financiero solo Movies",
        ],
    }
)

ep1_availability

,variable,disponible_movies,disponible_tv_shows,uso_previsto
0,type,True,True,comparación de contenido
1,country,True,True,análisis geográfico
2,release_year,True,True,evolución temporal
3,genres,True,True,caracterización por género
4,language,True,True,caracterización por idioma
5,popularity,True,True,índice relativo de popularidad
6,vote_average,True,True,valoración en escala 0–10
7,vote_count,True,True,respaldo de la valoración
8,budget,True,False,análisis financiero solo Movies
9,revenue,True,False,análisis financiero solo Movies


## 14. Inspección de variables categóricas y potencialmente multivalor

Se muestran ejemplos, nulos y una señal de presencia de comas para `country`, `genres`, `language`, `director` y `cast`. La señal solo ayuda a detectar posibles valores multivalor; no se separan categorías, actores ni directores.

In [68]:
variables_categoricas = ["country", "genres", "language", "director", "cast"]

def categorical_snapshot(dataframe: pd.DataFrame, source_name: str, column: str) -> dict[str, object]:
    if column not in dataframe.columns:
        return {
            "fuente": source_name,
            "variable": column,
            "disponible": False,
            "tipo": pd.NA,
            "nulos": pd.NA,
            "porcentaje_nulo": pd.NA,
            "ejemplos_observados": pd.NA,
            "celdas_con_coma": pd.NA,
        }

    observed_values = dataframe[column]
    examples = "; ".join(
        "<NA>" if pd.isna(value) else str(value)
        for value in observed_values.head(3)
    )
    return {
        "fuente": source_name,
        "variable": column,
        "disponible": True,
        "tipo": str(dataframe[column].dtype),
        "nulos": int(observed_values.isna().sum()),
        "porcentaje_nulo": observed_values.isna().mean() * 100,
        "ejemplos_observados": examples,
        "celdas_con_coma": int(
            observed_values.map(
                lambda value: "," in str(value) if pd.notna(value) else False
            ).sum()
        ),
    }

categorical_snapshot_table = pd.DataFrame(
    [
        categorical_snapshot(dataframe, source_name, column)
        for dataframe, source_name in ((movies_df, "Movies"), (tv_shows_df, "TV Shows"))
        for column in variables_categoricas
    ]
)

categorical_snapshot_table

,fuente,variable,disponible,tipo,nulos,porcentaje_nulo,ejemplos_observados,celdas_con_coma
0,Movies,country,True,str,466,2.91250,"United States of America; United Kingdom, United States of America; United Kingdom, United States of America",4058
1,Movies,genres,True,str,107,0.66875,"Comedy, Adventure, Fantasy, Animation, Family; Action, Science Fiction, Adventure; Adventure, Fantasy",12116
2,Movies,language,True,str,0,0.00000,en; en; en,0
3,Movies,director,True,str,132,0.82500,Mike Mitchell; Christopher Nolan; David Yates,1286
4,Movies,cast,True,str,204,1.27500,"Mike Myers, Eddie Murphy, Cameron Diaz, Antonio Banderas, Walt Dohrn; Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...",15632
5,TV Shows,country,True,str,1797,11.23125,South Korea; United States of America; Greece,1307
6,TV Shows,genres,True,str,974,6.08750,"Comedy, Reality; Talk, Comedy, News; Reality",8696
7,TV Shows,language,True,str,0,0.00000,ko; en; el,0
8,TV Shows,director,True,str,10965,68.53125,안재철; <NA>; <NA>,1321
9,TV Shows,cast,True,str,1157,7.23125,"Yoo Jae-suk, Jee Seok-jin, Kim Jong-kook, Haha, Song Ji-hyo; Conan O'Brien, Andy Richter; <NA>",13353


## 15. Hallazgos preliminares y traspaso a etapas posteriores

La siguiente síntesis se construye a partir de las validaciones ejecutadas. Describe evidencias observadas y su impacto potencial, sin aplicar correcciones.

In [69]:
date_types = dtype_comparison.loc[
    dtype_comparison["variable"] == "date_added", ["tipo_movies", "tipo_tv"]
].iloc[0]

date_observation = (
    "date_added no está almacenada como datetime en al menos una fuente."
    if any(dtype != "datetime64[ns]" for dtype in date_types)
    else "date_added está almacenada como datetime en ambas fuentes."
)

date_added_parseable = (date_added_validation_table["no_interpretable"] == 0).all()
date_added_complete = (date_added_validation_table["nulos"] == 0).all()
date_added_observation = (
    "date_added está presente, sin nulos y todos sus valores son interpretables como fecha en el diagnóstico; se mantiene como texto en la fuente."
    if date_added_parseable and date_added_complete
    else "date_added requiere revisar disponibilidad o valores no interpretables según la tabla de diagnóstico."
)

release_year_complete = (release_year_validation_table["nulos"] == 0).all()
release_year_observation = (
    f"release_year está disponible como {release_year_validation_table['tipo_observado'].iloc[0]}, sin nulos y con rango {release_year_validation_table['minimo'].min()}–{release_year_validation_table['maximo'].max()}."
    if release_year_complete
    else "release_year presenta valores nulos; la tabla muestra el detalle por fuente."
)

id_issue = any(
    value != 0
    for value in show_id_validation["diferencia_registros_vs_unicos"]
    if pd.notna(value)
)
id_observation = (
    "Hay diferencia entre registros e identificadores únicos en al menos una fuente."
    if id_issue
    else "La cantidad de show_id únicos coincide con los registros en ambas fuentes."
)

multivalue_issue = any(
    value > 0
    for value in categorical_snapshot_table["celdas_con_coma"]
    if pd.notna(value)
)
multivalue_observation = (
    "Se observan celdas con comas en al menos una variable categórica inspeccionada."
    if multivalue_issue
    else "No se observaron comas en la muestra estructural de variables categóricas."
)

financial_available = set(["budget", "revenue"]).issubset(columnas_movies) and not (
    set(["budget", "revenue"]) & columnas_tv
)
financial_observation = (
    "budget y revenue están disponibles solo en Movies."
    if financial_available
    else "La disponibilidad de budget y revenue difiere de la expectativa del caso."
)

hallazgos_base = pd.DataFrame(
    [
        {
            "aspecto": "Estructura",
            "observacion": f"Movies: {movies_df.shape[0]:,}×{movies_df.shape[1]}; TV Shows: {tv_shows_df.shape[0]:,}×{tv_shows_df.shape[1]}.",
            "impacto": "Delimita el tamaño y las fuentes a comparar.",
            "proxima_etapa": "feature/calidad-datos",
        },
        {
            "aspecto": "Columnas comparables",
            "observacion": f"Se identificaron {len(columnas_comunes)} columnas comunes, {len(solo_movies)} exclusivas de Movies y {len(solo_tv)} exclusivas de TV Shows.",
            "impacto": "Las comparaciones futuras deben limitarse a variables equivalentes.",
            "proxima_etapa": "feature/preparacion-catalogo",
        },
        {
            "aspecto": "Variables financieras",
            "observacion": financial_observation,
            "impacto": "Los análisis de budget, revenue y ROI deben restringirse a Movies.",
            "proxima_etapa": "feature/preparacion-catalogo",
        },
        {
            "aspecto": "Fechas",
            "observacion": date_observation,
            "impacto": "Se debe revisar formato, valores faltantes y conversión documentada si corresponde.",
            "proxima_etapa": "feature/calidad-datos",
        },
        {
            "aspecto": "Identificadores",
            "observacion": id_observation,
            "impacto": "La unicidad de show_id debe validarse antes de cualquier deduplicación.",
            "proxima_etapa": "feature/calidad-datos",
        },
        {
            "aspecto": "Variables multivalor",
            "observacion": multivalue_observation,
            "impacto": "Country, genres y language pueden requerir una estrategia documentada de normalización posterior.",
            "proxima_etapa": "feature/preparacion-catalogo",
        },
    ]
)

rating_matches_all = all(
    percentage == 100 for percentage in rating_validation["porcentaje_coincidencia"]
)
rating_observation = (
    "rating coincide completamente con vote_average en ambas fuentes; esto contradice su significado categórico documentado."
    if rating_matches_all
    else "La coincidencia entre rating y vote_average requiere revisión detallada; la tabla muestra la evidencia por fuente."
)

movies_duration = duration_validation_table.loc[duration_validation_table["fuente"] == "Movies"].iloc[0]
tv_duration = duration_validation_table.loc[duration_validation_table["fuente"] == "TV Shows"].iloc[0]
duration_observation = (
    f"Movies tiene {movies_duration['no_nulos']:,} valores no nulos de duration; TV Shows tiene {tv_duration['valores_unicos_no_nulos']:,} valor único no nulo."
)

vote_range_consistent = (vote_average_validation["fuera_rango_0_10"] == 0).all()
vote_observation = (
    "No se observaron valores de vote_average fuera del rango 0–10 en ninguna fuente."
    if vote_range_consistent
    else "Se observaron valores de vote_average fuera del rango 0–10; la tabla indica el conteo por fuente."
)

country_multivalue = categorical_snapshot_table.loc[categorical_snapshot_table["variable"] == "country", "celdas_con_coma"].sum() > 0
genres_multivalue = categorical_snapshot_table.loc[categorical_snapshot_table["variable"] == "genres", "celdas_con_coma"].sum() > 0
language_multivalue = categorical_snapshot_table.loc[categorical_snapshot_table["variable"] == "language", "celdas_con_coma"].sum() > 0
director_multivalue = categorical_snapshot_table.loc[categorical_snapshot_table["variable"] == "director", "celdas_con_coma"].sum() > 0
cast_multivalue = categorical_snapshot_table.loc[categorical_snapshot_table["variable"] == "cast", "celdas_con_coma"].sum() > 0

director_cast_observation = (
    f"director presenta {int(categorical_snapshot_table.loc[(categorical_snapshot_table['fuente'] == 'Movies') & (categorical_snapshot_table['variable'] == 'director'), 'nulos'].iloc[0]):,} nulos en Movies y {int(categorical_snapshot_table.loc[(categorical_snapshot_table['fuente'] == 'TV Shows') & (categorical_snapshot_table['variable'] == 'director'), 'nulos'].iloc[0]):,} en TV Shows; cast presenta {int(categorical_snapshot_table.loc[(categorical_snapshot_table['fuente'] == 'Movies') & (categorical_snapshot_table['variable'] == 'cast'), 'nulos'].iloc[0]):,} y {int(categorical_snapshot_table.loc[(categorical_snapshot_table['fuente'] == 'TV Shows') & (categorical_snapshot_table['variable'] == 'cast'), 'nulos'].iloc[0]):,}, respectivamente."
)
multivalue_observation = (
    "Se detectaron celdas con comas en country, genres, director y cast; no se detectaron en language bajo el separador analizado."
    if country_multivalue and genres_multivalue and director_multivalue and cast_multivalue and not language_multivalue
    else "La señal de multivalor debe interpretarse según los conteos observados en la tabla."
)

hallazgos_preliminares = pd.DataFrame(
    [
        {
            "categoria": "Conforme con el caso",
            "aspecto": "Dimensiones y estructura",
            "observacion": f"Movies: {movies_df.shape[0]:,}×{movies_df.shape[1]}; TV Shows: {tv_shows_df.shape[0]:,}×{tv_shows_df.shape[1]}.",
            "impacto": "La estructura general coincide con la referencia de tamaño del caso.",
            "proxima_etapa": "Sin acción inmediata",
        },
        {
            "categoria": "Conforme con el caso",
            "aspecto": "Variables financieras",
            "observacion": financial_observation,
            "impacto": "Los análisis de budget, revenue y ROI deben restringirse a Movies.",
            "proxima_etapa": "análisis posterior",
        },
        {
            "categoria": "Inconsistencia diccionario-datos",
            "aspecto": "rating",
            "observacion": rating_observation,
            "impacto": "No debe interpretarse como clasificación etaria sin investigar la discrepancia.",
            "proxima_etapa": "feature/calidad-datos",
        },
        {
            "categoria": "Conforme con el caso",
            "aspecto": "vote_average",
            "observacion": vote_observation,
            "impacto": "La escala observada es consistente con la regla 0–10.",
            "proxima_etapa": "feature/calidad-datos",
        },
        {
            "categoria": "Limitación de disponibilidad/calidad",
            "aspecto": "duration",
            "observacion": duration_observation,
            "impacto": "La presencia de la columna no garantiza utilidad para comparaciones de duración.",
            "proxima_etapa": "feature/calidad-datos",
        },
        {
            "categoria": "Inconsistencia diccionario-datos",
            "aspecto": "show_id",
            "observacion": "El caso lo documenta como texto; el tipo observado se muestra como entero en ambas fuentes.",
            "impacto": "La representación adecuada del identificador debe definirse sin tratarlo como medida numérica.",
            "proxima_etapa": "feature/calidad-datos",
        },
        {
            "categoria": "Limitación de disponibilidad/calidad",
            "aspecto": "Unicidad de show_id",
            "observacion": id_observation,
            "impacto": "La señal de posible duplicidad requiere investigación antes de deduplicar.",
            "proxima_etapa": "feature/calidad-datos",
        },
        {
            "categoria": "Inconsistencia diccionario-datos",
            "aspecto": "date_added",
            "observacion": date_observation,
            "impacto": "La interpretación como fecha requiere revisión de formato posterior.",
            "proxima_etapa": "feature/calidad-datos",
        },
        {
            "categoria": "Limitación de disponibilidad/calidad",
            "aspecto": "country y genres",
            "observacion": (
                "Se detectaron celdas con comas en country y genres."
                if country_multivalue and genres_multivalue
                else "La señal de multivalor debe revisarse según la evidencia de la tabla."
            ),
            "impacto": "Requerirán una estrategia posterior para análisis por categoría.",
            "proxima_etapa": "feature/preparacion-catalogo",
        },
        {
            "categoria": "Conforme con la evidencia observada",
            "aspecto": "language",
            "observacion": (
                "Se detectaron celdas con comas en language."
                if language_multivalue
                else "No se detectaron comas en language bajo el separador analizado."
            ),
            "impacto": "No se asume una representación multivalor sin evidencia adicional.",
            "proxima_etapa": "feature/preparacion-catalogo",
        },
    ]
)

hallazgos_preliminares = pd.DataFrame(
    [
        {
            "aspecto": "Dimensiones y estructura",
            "observacion": f"Movies: {movies_df.shape[0]:,}×{movies_df.shape[1]}; TV Shows: {tv_shows_df.shape[0]:,}×{tv_shows_df.shape[1]}.",
            "clasificacion": "Conforme con el caso",
            "impacto": "La estructura general coincide con la referencia de tamaño del caso.",
            "proxima_etapa": "Sin acción inmediata",
        },
        {
            "aspecto": "Variables financieras",
            "observacion": financial_observation,
            "clasificacion": "Conforme con el caso",
            "impacto": "Los análisis de budget, revenue y ROI deben restringirse a Movies.",
            "proxima_etapa": "análisis posterior",
        },
        {
            "aspecto": "date_added",
            "observacion": date_added_observation,
            "clasificacion": "Requiere preparación posterior",
            "impacto": "La fuente parece apta para prepararse para análisis temporal sin modificarla en esta etapa.",
            "proxima_etapa": "feature/preparacion-catalogo",
        },
        {
            "aspecto": "release_year",
            "observacion": release_year_observation,
            "clasificacion": "Conforme con el caso",
            "impacto": "Está estructuralmente disponible para análisis temporal posterior.",
            "proxima_etapa": "Sin acción inmediata",
        },
        {
            "aspecto": "rating",
            "observacion": rating_observation,
            "clasificacion": "Inconsistencia diccionario-datos",
            "impacto": "No debe interpretarse como clasificación etaria sin investigar la discrepancia.",
            "proxima_etapa": "feature/calidad-datos",
        },
        {
            "aspecto": "Clasificación etaria",
            "observacion": (
                "La fuente no contiene mediante rating la clasificación etaria descrita en el diccionario."
                if rating_matches_all
                else "La disponibilidad de una clasificación etaria confiable requiere investigación."
            ),
            "clasificacion": "Limitación de disponibilidad/calidad",
            "impacto": "No es posible reproducir confiablemente análisis de clasificación etaria con esta variable.",
            "proxima_etapa": "feature/calidad-datos",
        },
        {
            "aspecto": "vote_average",
            "observacion": vote_observation,
            "clasificacion": "Conforme con el caso",
            "impacto": "La escala observada es consistente con la regla 0–10.",
            "proxima_etapa": "Sin acción inmediata",
        },
        {
            "aspecto": "duration",
            "observacion": duration_observation,
            "clasificacion": "Limitación de disponibilidad/calidad",
            "impacto": "La presencia de la columna no garantiza utilidad para comparaciones de duración.",
            "proxima_etapa": "feature/calidad-datos",
        },
        {
            "aspecto": "show_id",
            "observacion": "El caso lo documenta como texto; el tipo observado se muestra como entero en ambas fuentes.",
            "clasificacion": "Inconsistencia diccionario-datos",
            "impacto": "La representación adecuada del identificador debe definirse sin tratarlo como medida numérica.",
            "proxima_etapa": "feature/calidad-datos",
        },
        {
            "aspecto": "Unicidad de show_id",
            "observacion": id_observation,
            "clasificacion": "Limitación de disponibilidad/calidad",
            "impacto": "La señal de posible duplicidad requiere investigación antes de deduplicar.",
            "proxima_etapa": "feature/calidad-datos",
        },
        {
            "aspecto": "director y cast",
            "observacion": director_cast_observation,
            "clasificacion": "Limitación de disponibilidad/calidad",
            "impacto": "Los nulos deben evaluarse antes de análisis de producción.",
            "proxima_etapa": "feature/calidad-datos",
        },
        {
            "aspecto": "Variables multivalor",
            "observacion": multivalue_observation,
            "clasificacion": "Requiere preparación posterior",
            "impacto": "La preparación deberá considerar country, genres, director y cast, sin asumir multivalor para language.",
            "proxima_etapa": "feature/preparacion-catalogo",
        },
    ]
)

hallazgos_preliminares

,aspecto,observacion,clasificacion,impacto,proxima_etapa
0,Dimensiones y estructura,"Movies: 16,000×18; TV Shows: 16,000×16.",Conforme con el caso,La estructura general coincide con la referencia de tamaño del caso.,Sin acción inmediata
1,Variables financieras,budget y revenue están disponibles solo en Movies.,Conforme con el caso,"Los análisis de budget, revenue y ROI deben restringirse a Movies.",análisis posterior
2,date_added,"date_added está presente, sin nulos y todos sus valores son interpretables como fecha en el diagnóstico; se mantiene...",Requiere preparación posterior,La fuente parece apta para prepararse para análisis temporal sin modificarla en esta etapa.,feature/preparacion-catalogo
3,release_year,"release_year está disponible como int64, sin nulos y con rango 2010–2025.",Conforme con el caso,Está estructuralmente disponible para análisis temporal posterior.,Sin acción inmediata
4,rating,rating coincide completamente con vote_average en ambas fuentes; esto contradice su significado categórico documentado.,Inconsistencia diccionario-datos,No debe interpretarse como clasificación etaria sin investigar la discrepancia.,feature/calidad-datos
5,Clasificación etaria,La fuente no contiene mediante rating la clasificación etaria descrita en el diccionario.,Limitación de disponibilidad/calidad,No es posible reproducir confiablemente análisis de clasificación etaria con esta variable.,feature/calidad-datos
6,vote_average,No se observaron valores de vote_average fuera del rango 0–10 en ninguna fuente.,Conforme con el caso,La escala observada es consistente con la regla 0–10.,Sin acción inmediata
7,duration,Movies tiene 0 valores no nulos de duration; TV Shows tiene 1 valor único no nulo.,Limitación de disponibilidad/calidad,La presencia de la columna no garantiza utilidad para comparaciones de duración.,feature/calidad-datos
8,show_id,El caso lo documenta como texto; el tipo observado se muestra como entero en ambas fuentes.,Inconsistencia diccionario-datos,La representación adecuada del identificador debe definirse sin tratarlo como medida numérica.,feature/calidad-datos
9,Unicidad de show_id,Hay diferencia entre registros e identificadores únicos en al menos una fuente.,Limitación de disponibilidad/calidad,La señal de posible duplicidad requiere investigación antes de deduplicar.,feature/calidad-datos


## 16. Conclusiones

Las conclusiones siguientes se generan desde las validaciones anteriores. No incluyen conclusiones de negocio sobre géneros, países, popularidad o rendimiento financiero; esos análisis corresponden a etapas posteriores.

In [70]:
variables_ep1_disponibles = ep1_availability.apply(
    lambda row: row["disponible_movies"] or row["disponible_tv_shows"], axis=1
).sum()

print("Conclusiones de la exploración inicial")
print("Conforme con el caso:")
print(f"- Las dimensiones observadas son consistentes con el caso y existen {len(columnas_comunes)} variables estructuralmente compartidas.")
print(f"- {release_year_observation}")
print(f"- {vote_observation}")
print("Inconsistencias diccionario-datos:")
print(f"- {rating_observation}")
print("- show_id se documenta como texto, pero se observa como entero en ambas fuentes.")
print("Limitaciones de disponibilidad/calidad:")
print("- La fuente no dispone mediante rating de una clasificación etaria confiable; no es posible reproducir ese análisis con esta variable.")
print(f"- {duration_observation}")
print(f"- {director_cast_observation}")
print("Aspectos para feature/calidad-datos:")
print("- Investigar rating, representación y unicidad de show_id, disponibilidad de duration y nulos de director/cast.")
print("Aspectos para feature/preparacion-catalogo:")
print(f"- {date_added_observation}")
print(f"- {multivalue_observation}")
print(f"- Las variables consideradas para EP1 están estructuralmente presentes en las fuentes correspondientes para {variables_ep1_disponibles} de {len(variables_ep1)} requerimientos; su presencia no garantiza por sí sola disponibilidad, calidad ni comparabilidad.")
print("- No se alteraron los CSV RAW, no se crearon datos procesados y no se aplicaron transformaciones definitivas.")

Conclusiones de la exploración inicial
Conforme con el caso:
- Las dimensiones observadas son consistentes con el caso y existen 16 variables estructuralmente compartidas.
- release_year está disponible como int64, sin nulos y con rango 2010–2025.
- No se observaron valores de vote_average fuera del rango 0–10 en ninguna fuente.
Inconsistencias diccionario-datos:
- rating coincide completamente con vote_average en ambas fuentes; esto contradice su significado categórico documentado.
- show_id se documenta como texto, pero se observa como entero en ambas fuentes.
Limitaciones de disponibilidad/calidad:
- La fuente no dispone mediante rating de una clasificación etaria confiable; no es posible reproducir ese análisis con esta variable.
- Movies tiene 0 valores no nulos de duration; TV Shows tiene 1 valor único no nulo.
- director presenta 132 nulos en Movies y 10,965 en TV Shows; cast presenta 204 y 1,157, respectivamente.
Aspectos para feature/calidad-datos:
- Investigar rating, represe